Before writing any sort of application, CRUD functions need to be developed and tested. This notebook provides a space for that.

In [5]:
import pandas as pd
from sqlalchemy import create_engine, text

In [6]:
# Running this cell REQUIRES a module named con_lib1.py in this notebook's PARENT DIRECTORY
# con_lib1.py is a copy of con_EXAMPLE.py with schema = "sample_library1"

import sys
sys.path.append("..") # con_lib1 is in the parent directory of this notebook
from con_lib1 import connection_string


In [7]:
engine = create_engine(connection_string)

# Create

## Define

In [8]:
def create_friend(name, max_loans=2, notes=None):
    df = pd.DataFrame(
        [[name, max_loans, notes]],
        columns=["name", "max_loans", "notes"]
    )
    df.to_sql(
        "friends", 
        if_exists="append", 
        con=connection_string, 
        index=False
    )
    message = f"Added '{name}' to 'friends'."
    
    return message

In [9]:
def create_book(title, isbn, author=None, genre=None):
    df = pd.DataFrame(
        [[title, author, genre, isbn]], 
        columns=["title", "author", "genre", "isbn"]
    )
    df.to_sql(
        "books",
        if_exists="append", 
        con=connection_string, 
        index=False
    )
    message = f"Added '{title}' to 'books'."
        
    return message

In [10]:
def create_loan(friend, book, loan_date=pd.Timestamp.today().date(), next_contact=pd.Timestamp.today().date() + pd.Timedelta(30, "D"), notes=None):
    df = pd.DataFrame(
        [[book["isbn"], friend["friend_id"], loan_date, next_contact, notes]],
        columns = ["isbn", "friend_id", "loan_date", "next_contact", "notes"]
    )
    df.to_sql(
        "loans",
        if_exists="append",
        con=connection_string,
        index=False
    )

    # keep books.is_available in sync -- book just went out on loan
    with engine.begin() as connection:
        connection.execute(
            text("UPDATE books SET is_available = FALSE WHERE isbn = :isbn"),
            {"isbn": book["isbn"]}
        )

    message = f"Added '{friend["name"]}' borrowed '{book["title"]}' to 'loans'."

    return message


## Test

In [11]:
pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Priya Nair,3,"Prefers thrillers, returns on time"
1,2,Tom Becker,3,"Bit slow with returns, send reminders"
2,3,Meera Iyer,2,NaN
3,4,Jonas Weber,1,Only borrows non-fiction
4,6,Soso,2,"Doesn't answer phone, use socials."
5,8,Ed,2,NaN
6,9,Eddy,2,NaN
7,10,Edd,2,Not sure he can read
8,11,Ed,2,NaN
9,12,Eddy,1,NaN


In [12]:
create_friend('Anna')
create_friend("Essa", 1)
create_friend("Alen", notes="Not sure he can read")

pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Priya Nair,3,"Prefers thrillers, returns on time"
1,2,Tom Becker,3,"Bit slow with returns, send reminders"
2,3,Meera Iyer,2,NaN
3,4,Jonas Weber,1,Only borrows non-fiction
4,6,Soso,2,"Doesn't answer phone, use socials."
5,8,Ed,2,NaN
6,9,Eddy,2,NaN
7,10,Edd,2,Not sure he can read
8,11,Ed,2,NaN
9,12,Eddy,1,NaN


In [13]:
create_book("Words on a Page", "0000000000000")
create_book("Alice's Big Nap", "0000000000", "A. Snooze"),
create_book("More Words on a Page", "1234567890", genre="Book")

pd.read_sql("books", con=connection_string)

DatabaseError: Execution failed on sql 'INSERT INTO books (title, author, genre, isbn) VALUES (:title, :author, :genre, :isbn)': (pymysql.err.IntegrityError) (1062, "Duplicate entry '0000000000000' for key 'books.PRIMARY'")
[SQL: INSERT INTO books (title, author, genre, isbn) VALUES (%(title)s, %(author)s, %(genre)s, %(isbn)s)]
[parameters: {'title': 'Words on a Page', 'author': None, 'genre': None, 'isbn': '0000000000000'}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [14]:
book_result = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780143127550'", con=connection_string)
if book_result.empty:
    print("No book found with that ISBN")
else:
    book = book_result.iloc[0]


In [ ]:
friend =  pd.read_sql("SELECT * FROM friends WHERE friend_id = 1", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780143127550'", con=connection_string).iloc[0]
create_loan(friend, book)
             
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 2", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780061120084'", con=connection_string).iloc[0]
loan_date = '2026-01-01'
create_loan(friend, book, loan_date)
             
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 3", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780307474278'", con=connection_string).iloc[0]
next_contact='2026-02-15'
create_loan(friend, book, next_contact=next_contact)
             
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 4", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780142437230'", con=connection_string).iloc[0]
notes="Reading for thesis."
create_loan(friend, book, notes=notes)


pd.read_sql("loans", con=connection_string)


,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,1,2026-01-01,NaT,2026-10-14,NaN
1,9780143127741,5,2026-09-14,NaT,2026-10-14,Reading for thesis.
2,9780385490818,2,2026-09-14,NaT,2026-02-15,NaN
3,9780987654321,5,2025-07-02,2025-07-02,2025-07-25,NaN
4,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
5,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,NaN
6,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,NaN
7,9785566778899,6,2026-09-14,NaT,2026-10-14,NaN


# Read

## Define

In [ ]:
def prettify_df(df):
    df.columns = [c.upper() if c == "isbn" else c.replace("_", " ").capitalize() for c in df.columns]
    return df.fillna("")

In [ ]:
def read_friends():
    return pd.read_sql("friends", con=connection_string)

def display_friends(friends):
    return friends.pipe(prettify_df).loc[:, "Name":]

In [ ]:
def read_books(available_only=False):
    books = pd.read_sql("books", con=connection_string)
    if available_only:
        loans = pd.read_sql("loans", con=connection_string)
        books = pd.merge(books, loans, on="isbn", how="left").query("friend_id.isna()")[books.columns]
    return books

def display_books(books):
    return books.pipe(prettify_df).sort_values(by="Title")

In [15]:
def read_loans():
    return pd.read_sql("loans", con=connection_string)

def display_loans():
    friends = read_friends()
    books = read_books()
    loans = read_loans()
    for c in ["loan_date", "last_contact", "next_contact"]:
        loans[c] = loans[c].dt.strftime("%Y-%m-%d")

    display_columns = ["title", "name", "loan_date", "last_contact", "next_contact", "notes"]
    df = (
        pd.merge(loans, friends, on="friend_id", suffixes=["", "_no"])
        .merge(books, on="isbn")
        [display_columns]
    )
    return df.pipe(prettify_df).sort_values(by="Loan date")

## Test

In [ ]:
read_friends()
display_friends(read_friends())

,Name,Max loans,Notes
0,Priya Nair,2,"Prefers thrillers, returns on time"
1,Tom Becker,3,"Bit slow with returns, send reminders"
2,Meera Iyer,2,
3,Jonas Weber,1,Only borrows non-fiction
4,Ed,2,
5,Soso,2,"Doesn't answer phone, use socials."
6,Edd,2,Not sure he can read
7,Ed,2,
8,Eddy,1,
9,Edd,2,Not sure he can read


In [ ]:
read_books()
display_books(read_books())

,ISBN,Title,Author,Genre,Is available
8,9780553380163,A Brief History of Time,Stephen Hawking,Science,0
0,0000000000,Alice's Big Nap,A. Snooze,,1
7,9780441172719,Dune,Frank Herbert,Sci-Fi,0
4,9780142437230,Meditations,Marcus Aurelius,Philosophy,1
2,1234567890,More Words on a Page,,Book,1
6,9780307474278,The Da Vinci Code,Dan Brown,Thriller,1
5,9780143127550,The Goldfinch,Donna Tartt,Fiction,1
3,9780061120084,To Kill a Mockingbird,Harper Lee,Fiction,1
1,0000000000000,Words on a Page,,,1


In [ ]:
read_loans()
display_loans()

,Title,Name,Loan date,Last contact,Next contact,Notes
4,The Secret Ingredient,Luca Schmidt,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
5,The Paper Trail,Ellie Martinez,2025-06-15,2025-06-15,2025-07-15,
6,The Clockmaker's Son,Amira Jansen,2025-07-01,2025-06-30,2025-07-30,
3,Echoes of the Past,Fix Bauer,2025-07-02,2025-07-02,2025-07-25,
0,The Alchemist,Ellie Martinez,2026-01-01,,2026-10-14,
1,Sapiens: A Brief History of Humankind,Fix Bauer,2026-09-14,,2026-10-14,Reading for thesis.
2,The Poisonwood Bible,Davey,2026-09-14,,2026-02-15,
7,Gardens of Glass,Soso Klein,2026-09-14,,2026-10-14,


# Update

## Define

In [25]:
def format_field(field):
    if field == "dates":
        return "Contact dates"
    elif field == "isbn":
        return "ISBN"
    else:
        return field.replace("_", " ").capitalize()

In [24]:
def update_friend(friend, field, new_data):
    update_query = f"""UPDATE friends 
    SET {field} = '{new_data}' 
    WHERE friend_id = {friend["friend_id"]};"""
    with engine.begin() as connection:
        connection.execute(text(update_query))
        return f"{format_field(field)} updated."

In [23]:
def update_book(book, field, new_data):
    update_query = f"""UPDATE books
    SET {field} = '{new_data}'
    WHERE isbn = {book["isbn"]};"""
    with engine.begin() as connection:
        connection.execute(text(update_query))
        return f"{format_field(field)} updated."

In [22]:
def update_loan(loan, field, new_data):
    update_query = f"""UPDATE loans 
    SET {field} = '{new_data}' 
    WHERE loan_id = {loan["loan_id"]};"""
    if field == 'dates':
        update_query = f"""UPDATE loans
        SET last_contact = '{new_data[0]}', next_contact = '{new_data[1]}'
        WHERE loan_id = {loan["loan_id"]};"""
    with engine.begin() as connection:
        connection.execute(text(update_query))
        return f"{format_field(field)} updated."


## Test

In [ ]:
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 1", con=engine).iloc[0] # Priya Nair

update_friend(friend, 'name', 'Priya')
update_friend(friend, 'max_loans', 2)
update_friend(friend, 'notes', "Doesn't answer phone, use socials.".replace("'", "\\'")) 

pd.read_sql("friends", con=connection_string)


,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,NaN
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,NaN
4,5,Fix Bauer,3,NaN
5,6,Soso,2,"Doesn't answer phone, use socials."
6,7,Ed,2,NaN
7,8,Eddy,1,NaN
8,9,Edd,2,Not sure he can read


In [ ]:
book = pd.read_sql("SELECT * FROM books WHERE isbn = '9780142437230'", con=engine).iloc[0] # Meditations

update_book(book, 'title', 'Meditations (Revised Edition)')
update_book(book, 'author', 'Marcus Aurelius')
update_book(book, 'genre', 'Philosophy')
update_book(book, 'isbn', '9988776655879')


pd.read_sql("books", con=connection_string)


,title,author,genre,isbn
0,The Alchemist,Paulo Coelho,Fiction,9780062316110
1,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
2,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
3,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
4,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
5,Echoes of the Past,Julian Marsh,Thriller,9780987654321
6,The Secret Ingredient,Samira Nouri,Romance,9781122334455
7,The Paper Trail,Liane Forestier,Historical Fiction,9781234567890
8,The Clockmaker's Son,Hugo Vernier,Steampunk,9784455667788
9,Gardens of Obsidian,Poison Ivy,Romantasy,9988776655879


In [26]:
loan = pd.read_sql("SELECT * FROM loans WHERE isbn = '9780441172719' AND friend_id = 2", con=engine).iloc[0] # Dune, borrowed by Tom Becker

today = pd.Timestamp.today().date()
next_week = today + pd.Timedelta(1, "w")
update_loan(loan, 'notes', 'Got caught in rainstorm with book')
update_loan(loan, 'dates', (today, next_week))

pd.read_sql("loans", con=connection_string)


/var/folders/g9/hf05tqzd2296n5c4sdm7g82w0000gn/T/ipykernel_60618/2070258470.py:4: Pandas4Warning: 'w' is deprecated and will be removed in a future version. Please use 'W' instead of 'w'.
  next_week = today + pd.Timedelta(1, "w")


,loan_id,isbn,friend_id,loan_date,return_date,last_contact,next_contact,notes
0,1,9780143127550,1,2026-05-10,2026-05-28,NaT,NaT,"Loved it, asked for similar recs"
1,2,9780441172719,3,2026-04-02,2026-04-20,NaT,NaT,NaN
2,3,9780061120084,2,2026-08-05,NaT,2026-09-05,2026-09-19,"Reminder sent, no response yet"
3,4,9780307474278,4,2026-03-15,2026-04-01,NaT,NaT,Returned a bit late
4,5,9780142437230,3,2026-08-28,NaT,NaT,2026-09-27,NaN
5,7,9780143127550,2,2026-06-01,2026-06-18,NaT,NaT,NaN
6,8,9780441172719,4,2026-01-05,2026-01-22,NaT,NaT,First loan for Jonas
7,10,9780142437230,2,2026-07-10,2026-07-30,NaT,NaT,NaN
8,11,9780441172719,2,2026-08-15,NaT,2026-09-16,2026-09-23,Got caught in rainstorm with book
9,13,9780307474278,1,2026-07-01,2026-07-20,NaT,NaT,Returned in great condition


# Delete

## Define

In [16]:
def delete_friend(friend):
    delete_query = f"""DELETE FROM friends
    WHERE friend_id = '{friend["friend_id"]}';"""
    with engine.begin() as connection:
        connection.execute(text(delete_query))
        return f"Removed '{friend['name']}' from 'friends'."

In [17]:
def delete_book(book):
    delete_query = f"""DELETE FROM books
    WHERE isbn = '{book["isbn"]}';"""
    with engine.begin() as connection:
        connection.execute(text(delete_query))
        return f"Removed '{book['title']}' from 'books'."

In [ ]:
def delete_loan(loan):
    

In [18]:
def delete_loan(loan):
    lookup_table = (pd.read_sql("loans", con=connection_string).loc[[loan.name]]
                    .merge(pd.read_sql("friends", con=connection_string), on="friend_id")
                    .merge(pd.read_sql("books", con=connection_string), on="isbn")
                    .iloc[0]
                   )
    delete_query = f"""DELETE FROM loans
    WHERE loan_id = {lookup_table["loan_id"]};"""
    with engine.begin() as connection:
        connection.execute(text(delete_query))
        if pd.isna(lookup_table.get("return_date")):
            connection.execute(
                text("UPDATE books SET is_available = TRUE WHERE isbn = :isbn"),
                {"isbn": lookup_table["isbn"]}
            )
        return f"Removed '{lookup_table["name"]}' borrowed '{lookup_table["title"]}' from 'loans'."


## Test

In [ ]:
pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,None
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,None
4,5,Fix Bauer,3,None
5,6,Soso,2,"Doesn't answer phone, use socials."
6,7,Ed,2,None
7,8,Eddy,1,None
8,9,Edd,2,Not sure he can read


In [ ]:
friend = pd.read_sql("friends", con=connection_string).iloc[-1]
delete_friend(friend)

"Removed 'Edd' from 'friends'."

In [ ]:
pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,None
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,None
4,5,Fix Bauer,3,None
5,6,Soso,2,"Doesn't answer phone, use socials."
6,7,Ed,2,None
7,8,Eddy,1,None


In [ ]:
pd.read_sql("books", con=connection_string)

,title,author,genre,isbn
0,Alice's Big Nap,A. Snooze,None,0000000000
1,Words on a Page,None,None,0000000000000
2,More Words on a Page,None,Book,1234567890
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
5,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
7,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


In [ ]:
book = pd.read_sql("books", con=connection_string).iloc[-1]
delete_book(book)

"Removed 'A Brief History of Time' from 'books'."

In [ ]:
pd.read_sql("books", con=connection_string)

,title,author,genre,isbn
0,Alice's Big Nap,A. Snooze,None,0000000000
1,Words on a Page,None,None,0000000000000
2,More Words on a Page,None,Book,1234567890
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
5,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
7,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


Test comparison of table before and after delete and see if the 'missing row' matches the orginally specified loan

In [ ]:
table_pre = pd.read_sql("loans", con=connection_string)

In [ ]:
loan = pd.read_sql("loans", con=connection_string).iloc[-1]
delete_loan(loan)

"Removed 'Amira Jansen' borrowed 'The Clockmaker's Son' from 'loans'."

In [ ]:
table_post = pd.read_sql("loans", con=connection_string)

In [ ]:
dropped_line = pd.concat([table_pre, table_post]).drop_duplicates(keep=False).iloc[0]
dropped_line

isbn                  9784455667788
friend_id                         4
loan_date       2025-07-01 00:00:00
last_contact    2025-06-30 00:00:00
next_contact    2025-07-30 00:00:00
notes                          None
Name: 6, dtype: object

In [ ]:
loan.equals(dropped_line)

True

# Validate

## Define

In [27]:
def validate_name(name):
    if (name is None) or (not name.strip()):
        return "Warning. Empty name not accepted."
    else:
        return ""

In [28]:
def validate_isbn(isbn):
    if len(isbn) not in (10, 13):
        return "Warning. ISBN must be 10 or 13 digits long."
    elif not isbn.isnumeric():
        return "Warning. ISBN must be numeric."
    elif isbn in pd.read_sql("SELECT isbn FROM books", con=connection_string)["isbn"].values:
        return "Warning. This ISBN is already in use."
    else:
        return ""

In [29]:
def validate_title(title):
    if (title is None) or (not title.strip()):
        return "Warning. Empty title not accepted."
    else:
        return ""

In [30]:
def validate_loan_taker(friend):
    current_loans = pd.read_sql("loans", con=connection_string)
    if friend["friend_id"] in current_loans["friend_id"].unique():
        num_loans = current_loans.value_counts("friend_id").loc[friend["friend_id"]]
        if friend["max_loans"] == num_loans:
            return f"Warning. {friend["name"]} has already reached their maximum loan allowance."
    else:
        return ""

In [ ]:
def validate_loan_item(book):
    current_loans = pd.read_sql("loans", con=connection_string)
    if book["isbn"] in current_loans["isbn"].values:
        return f"Warning. {book["title"]} is already on loan."
    else:
        return ""

## Test

In [ ]:
validate_name("Xena")
validate_name("")
validate_name("    ")
validate_name(None)

'Warning. Empty name not accepted.'

In [ ]:
validate_isbn("1111111111111")
validate_isbn("00001")
validate_isbn("000000000a")
validate_isbn("9780307949486")

'Warning. This ISBN is already in use.'

In [ ]:
validate_title("This is Not a Title")
validate_title("")
validate_title("    ")
validate_title(None)

'Warning. Empty title not accepted.'

In [ ]:
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 6", con=connection_string).iloc[0]
validate_loan_taker(friend)

friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 3", con=connection_string).iloc[0]
validate_loan_taker(friend)

'Warning. Luca Schmidt has already reached their maximum loan allowance.'

In [ ]:
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780307949486'", con=connection_string).iloc[0] # The Wind-Up Bird Chronicle 
validate_loan_item(book)

book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9781122334455'", con=connection_string).iloc[0] # The Secret Ingredient
validate_loan_item(book)

'Warning. The Secret Ingredient is already on loan.'